In [7]:
from typing import TypedDict

from langgraph.constants import START, END
from langgraph.graph import StateGraph


# 输入状态
class InputState(TypedDict):
    username:str

# 输出状态
class OutputState(TypedDict):
    graph_out:str

# 全局状态
class OverAllState(TypedDict):
    username:str
    graph_out:str
    nickname:str

# 私有状态
class PrivateState(TypedDict):
    greeting:str


# 定义节点1(获取输入状态的上下文，修改状态内容到全局状态中)->OverAllState
def node_1(state:InputState)->OverAllState:
    # 向全局状态中添加username
    return {
        "nickname": "Dear"+state["username"]
    }

# 定义节点2（对接node1，使用全局状态中的上下文，修改状态内容到私有状态中）
def node_2(state:OverAllState)->PrivateState:
    # 向私有状态添加greeting
    return {
        "greeting": "Hello "+state["nickname"]
    }

# 定义节点3(对接node2 使用私有状态中的上下文，修改状态内容到输出状态中)
def node_3(state:PrivateState)->OutputState:
    # 向输出状态中添加graph_output
    return{
        "graph_out":state["greeting"]+" 很高兴认识你"
    }


# 构建图
builder= StateGraph(state_schema=OverAllState,input_schema=InputState,output_schema=OutputState)
# 添加节点
builder.add_node("node_1",node_1);
builder.add_node("node_2",node_2);
builder.add_node("node_3",node_3);

# 添加边
builder.add_edge(START,"node_1");
builder.add_edge("node_1","node_2");
builder.add_edge("node_2","node_3")
builder.add_edge("node_3",END);

# 获取图
graph = builder.compile()
# 运行图
result=graph.invoke({"username":"qingxun"})
print(result)









{'graph_out': 'Hello Dearqingxun 很高兴认识你'}
